# Treinamento ResNet + Lstm Multimodal

Será implementada uma nova loss baseada em BCE e Margin Ranking Loss.

## 1. Setup

In [ ]:
import os
import sys
import datetime
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

from transformers import AutoModel # transformers==4.44.0
import einops
import timm
import torchaudio # torchaudio==2.5.1

sys.path.insert(0, "/mnt/storage_C4/gaussian_football")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/preprocessing")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/models/nn")

from resnetlstm import ResNetLSTM
from resnetlstm_multimodal import ResNetLSTM_MultiModal
from loader_multimodal_pair import build_multimodal_dataloader, train_video_transform
#from balanced_dataset import balanced_df

In [2]:
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 435
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Torch: 2.5.1+cu121
Torchvision: 0.20.1+cu121
CUDA disponível: True
CUDA: 12.1
GPU: NVIDIA RTX A4500
Device: cuda


## 2. Configuração

In [3]:
# paths
LABELS_ALL   = "/mnt/storage_C4/gaussian_football/data/labels/labels_all.csv"
LABELS_DIR   = "/mnt/storage_C4/gaussian_football/data/labels"
CKPT_DIR     = "/mnt/storage_C4/gaussian_football/models/checkpoints"
RUNS_DIR     = "/mnt/storage_C4/gaussian_football/runs_multimodal_audio"  # logs do TensorBoard

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# treino 
EPOCHS        = 100        # teto por causa do early stopping decide
PATIENCE      = 20         # épocas sem melhora antes de parar
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0
MONITOR       = "loss"      # métrica de checkpoint/early-stopping: "ccc" | "mae" | "loss"

# dataloaders (mudei pra 1 pra rodar a resnet 50 c finetune)
BATCH_SIZE = 1

# modelo padrão
MODEL_DEFAULTS = dict(
    frame_step=2,
    hidden_size=256,
    num_layers=1,
    use_dropout=False,
    dropout_p=0.3,
)

## 3. Balanceando os Dados

Gera os CSVs balanceados (50% highlight / 50% não-highlight por partida,
com `threshold=0.5` no `arousal_score`). Só recalcula se os arquivos não existirem
(use `FORCE_REBUILD = True` para refazer).

In [4]:
FORCE_REBUILD = False

paths_balanced = {
    "train": os.path.join(LABELS_DIR, "balanced_labels_train.csv"),
    "valid": os.path.join(LABELS_DIR, "balanced_labels_val.csv"),
    "test":  os.path.join(LABELS_DIR, "balanced_labels_test.csv"),
}

if FORCE_REBUILD or not all(os.path.exists(p) for p in paths_balanced.values()):
    all_data = pd.read_csv(LABELS_ALL)
    for split, out_path in paths_balanced.items():
        subset = all_data[all_data["split"] == split]
        balanced = balanced_df(subset, "game_id", threshold=0.5, random_state=SEED)
        balanced.to_csv(out_path, index=False)
        print(f"{split}: {len(balanced)} clipes -> {out_path}")
else:
    print("CSVs balanceados já existem. (FORCE_REBUILD=True para refazer.)")

CSVs balanceados já existem. (FORCE_REBUILD=True para refazer.)


## 4. DataLoaders Multimodais

In [5]:
TRAIN_PATH = f"{LABELS_DIR}/balanced_labels_train.csv"
VAL_PATH   = f"{LABELS_DIR}/balanced_labels_val.csv"

train_multimodal_loader = build_multimodal_dataloader(
    csv_path=TRAIN_PATH,
    pair=True, # pareamento dos dados low e high
    split="train",
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
    video_transform=train_video_transform,
)

valid_multimodal_loader = build_multimodal_dataloader(
    csv_path=VAL_PATH,
    pair=True, # pareamento dos dados para loss marging ranking 
    split='valid',
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)

Dataset: 2942/3034 exemplos válidos.
Low: 1472
High: 1470
Dataset: 1177/1322 exemplos válidos.
Low: 589
High: 588


## 5. Métricas

#### CCC

O **CCC** (Concordance Correlation Coefficient) mede concordância entre predição e alvo,
punindo tanto erro de correlação quanto deslocamento de escala/média, por isso é sensível
ao *variance collapse* (modelo prevendo sempre perto da média).

$$\rho_c = \dfrac{2\,\rho\,\sigma_x\,\sigma_y}{\sigma_x^2 + \sigma_y^2 + (\mu_x - \mu_y)^2}$$

In [13]:
def calculate_ccc(y_true, y_pred):
    '''Concordance Correlation Coefficient sobre dois arrays 1D.'''
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true,  var_pred  = np.var(y_true),  np.var(y_pred)
    covariance = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    denominator = var_true + var_pred + (mean_true - mean_pred) ** 2
    return 0.0 if denominator == 0 else (2 * covariance) / denominator

### Perda Combinada

A função de perda usada irá combinar a BCE e a Margin Ranking Loss:

$$
Loss_{Total} = BCE_{Loss} + \lambda \cdot \text{Margin Ranking Loss}
$$

A Binary Cross Entropy vai penalizar diretamente desvios grandes na predição dos valores entre 0 e 1.

$$
BCE = - \frac{1}{N}\sum_{i=1}^{N}[y_i \cdot \log(\hat{y_i}) + (1-y_i)\cdot \log (1-\hat{y_i})]
$$

Enquanto o Margin Ranking Loss irá penalizar se o modelo atribuir baixa pontuação para amostras highlight e alta pontuação para amostras que não são highlights.

$$
Loss = \max (0, -Y \times (\hat{y}_{alto}- \hat{y}_{baixo}) + \text{margem})
$$

A margem utilizada será adaptativa, ou seja, para cada par de amostras a margem será calculada pela diferença das labels reais.

In [12]:
def adaptive_margin_ranking_loss(label_low, label_high, pred_low, pred_high, margin_scale=1.0):
    margin = margin_scale * (label_high - label_low)
    return torch.relu(margin - (pred_high - pred_low))

bce = nn.BCEWithLogitsLoss()

def combined_loss(label_low, label_high, pred_low, pred_high, alpha=1.0, margin_scale=1.0):
    loss_bce_low = bce(pred_low, label_low)
    loss_bce_high = bce(pred_high, label_high)
    loss_bce = 0.5 * (loss_bce_low + loss_bce_high)

    loss_rank = adaptive_margin_ranking_loss(label_low, label_high, pred_low, pred_high, margin_scale,)

    return loss_bce + alpha * loss_rank

In [7]:
class CombinedLoss(nn.Module):

    def __init__(self, alpha=1.0, margin_scale=1.0):
        super().__init__()

        self.alpha = alpha
        self.margin_scale = margin_scale
        self.bce = nn.BCEWithLogitsLoss()

    def forward(
        self,
        low_label,
        high_label,
        low_pred,
        high_pred,
        return_components=False,
    ):

        bce = (
            self.bce(low_pred, low_label)
            + self.bce(high_pred, high_label)
        ) / 2

        margin = self.margin_scale * (
            high_label - low_label
        )

        rank = torch.relu(
            margin - (high_pred - low_pred)
        ).mean()

        loss = bce + self.alpha * rank

        if return_components:
            return loss, bce, rank

        return loss

## 7. Treino

Função de treino unificada.

In [ ]:
MONITOR_MODE = {"ccc": "max", "mae": "min", "loss": "min"}

def _apply_freeze(model, freeze_backbone):
    '''Liga/desliga o gradiente da ResNet. conv1 (índice 0) fica sempre treinável.'''
    for name, param in model.cnn.named_parameters():
        param.requires_grad = name.startswith("0.") or (not freeze_backbone)

def _set_backbone_eval(model):
    '''Congela estatísticas de BatchNorm na ResNet. Conv1 fica em train.'''
    for i, module in enumerate(model.cnn):
        module.train() if i == 0 else module.eval()

def binary_accuracy(y_true, y_pred, threshold=0.5):
    return float(np.mean((y_pred > threshold) == (y_true > threshold)))

def _is_better(current, best, mode):
    return current > best if mode == "max" else current < best

def _log_pred_scatter(writer, y_true, y_pred, tag, step):
    '''Scatter pred x target — faixa horizontal achatada = variance collapse.'''
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(y_true, y_pred, s=8, alpha=0.4)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, "--", color="gray", linewidth=1)  # diagonal ideal
    ax.set_xlabel("target"); ax.set_ylabel("pred"); ax.set_title("pred vs target")
    writer.add_figure(tag, fig, step)
    plt.close(fig)


def train_model(
    model, train_loader, val_loader, *,
    criterion, run_name, optimizer, scheduler,
    backbone_name, loss_name,
    freeze_mode="finetune", unfreeze_epoch=5,
    freeze_bn_always=True,
    epochs=EPOCHS, patience=PATIENCE, monitor=MONITOR,
    grad_clip=GRAD_CLIP, use_amp=False,
    checkpoint_path=None, device=device,
):
    '''Treina e valida por época, logando tudo no TensorBoard.

    Args:
        criterion: função de perda (vindo de LOSSES).
        run_name: nome do experimento (usado no log_dir e no checkpoint).
        backbone_name, loss_name: usados nos hparams do TensorBoard.
        freeze_mode: "full" | "frozen" | "finetune".
        unfreeze_epoch: época de descongelamento (só vale para "finetune").
        monitor: métrica de checkpoint/early-stopping ("ccc" | "mae" | "loss").
        use_amp: ativa mixed precision (mais rápido; cheque o ccc_loss em fp16).
    Returns:
        dict com as melhores métricas e o caminho do checkpoint.
    '''
    model.to(device)
    torch.cuda.empty_cache()
    if checkpoint_path is None:
        checkpoint_path = os.path.join(CKPT_DIR, f"{run_name}.pth")
    last_path = os.path.join(CKPT_DIR, f"{run_name}_last.pth")

    mode = MONITOR_MODE[monitor]
    best_score = -np.inf if mode == "max" else np.inf
    best_metrics, best_true, best_pred = {}, None, None
    epochs_no_improve = 0

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join(RUNS_DIR, f"{run_name}_{stamp}")
    writer = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    for epoch in range(epochs):
        # estado de congelamento desta época
        if freeze_mode == "full":
            backbone_frozen = False
        elif freeze_mode == "frozen":
            backbone_frozen = True
        else:  # finetune
            backbone_frozen = epoch < unfreeze_epoch
            if epoch == unfreeze_epoch:
                print(f"[época {epoch+1}] descongelando a ResNet (fine-tuning completo)")

        _apply_freeze(model, backbone_frozen)

        # ----- treino -----
        model.train()
        if freeze_bn_always or backbone_frozen:
            _set_backbone_eval(model)

        train_loss = 0.0
        train_bce = 0.0
        train_rank = 0.0

        train_true, train_pred = [], []
        
        for (low, high) in tqdm(train_loader, desc=f"época {epoch+1}/{epochs} [treino]", leave=False):
            low_video, _, low_mel, low_targets = low
            high_video, _, high_mel, high_targets = high

            low_video = low_video.to(device, non_blocking=True)
            high_video = high_video.to(device, non_blocking=True)

            low_mel = low_mel.to(device, non_blocking=True)
            high_mel = high_mel.to(device, non_blocking=True)

            low_targets = low_targets.to(device).float().view(-1)
            high_targets = high_targets.to(device).float().view(-1)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda", enabled=use_amp):
                low_outputs = model(low_video, low_mel).reshape(-1)
                high_outputs = model(high_video, high_mel).reshape(-1)

                loss, bce_loss, rank_loss = criterion(
                    low_targets,
                    high_targets,
                    low_outputs,
                    high_outputs,
                    return_components=True,
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                grad_clip,
            )

            scaler.step(optimizer)
            scaler.update()

            batch_size = low_video.size(0)
            train_loss += loss.item() * batch_size
            train_bce += bce_loss.item() * batch_size
            train_rank += rank_loss.item() * batch_size

            preds = torch.cat([torch.sigmoid(low_outputs), torch.sigmoid(high_outputs)])
            targets = torch.cat([ low_targets,  high_targets])

            train_true.extend(targets.detach().cpu().numpy())
            train_pred.extend(preds.detach().cpu().numpy())
        
        train_loss /= len(train_loader.dataset)
        train_bce /= len(train_loader.dataset)
        train_rank /= len(train_loader.dataset)

        train_true, train_pred = np.array(train_true), np.array(train_pred)
        train_mae = np.mean(np.abs(train_true - train_pred))
        train_ccc = calculate_ccc(train_true, train_pred)
        train_acc = binary_accuracy(train_true, train_pred)

        # ----- validação -----
        model.eval()

        val_loss = 0.0
        val_bce = 0.0
        val_rank = 0.0

        all_true, all_pred = [], []

        with torch.no_grad():

            for (low, high) in tqdm(
                val_loader,
                desc=f"época {epoch+1}/{epochs} [val]",
                leave=False,
            ):

                low_video, _, low_mel, low_targets = low
                high_video, _, high_mel, high_targets = high

                low_video = low_video.to(device, non_blocking=True)
                high_video = high_video.to(device, non_blocking=True)

                low_mel = low_mel.to(device, non_blocking=True)
                high_mel = high_mel.to(device, non_blocking=True)

                low_targets = low_targets.to(device).float().view(-1)
                high_targets = high_targets.to(device).float().view(-1)

                with torch.amp.autocast("cuda", enabled=use_amp):

                    low_outputs = model(low_video, low_mel).reshape(-1)
                    high_outputs = model(high_video, high_mel).reshape(-1)

                    loss, bce_loss, rank_loss = criterion(
                        low_targets,
                        high_targets,
                        low_outputs,
                        high_outputs,
                        return_components=True,
                    )

                val_loss += loss.item() * low_video.size(0)
                val_bce += bce_loss.item() * low_video.size(0)
                val_rank += rank_loss.item() * low_video.size(0)

                preds = torch.cat([
                    torch.sigmoid(low_outputs),
                    torch.sigmoid(high_outputs),
                ])

                targets = torch.cat([
                    low_targets,
                    high_targets,
                ])

                all_true.extend(targets.detach().cpu().numpy())
                all_pred.extend(preds.detach().cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_bce /= len(val_loader.dataset)
        val_rank /= len(val_loader.dataset)

        all_true = np.array(all_true)
        all_pred = np.array(all_pred)
        
        pred_std = float(np.std(all_pred))
        target_std = float(np.std(all_true))
        val_mae = np.mean(np.abs(all_true - all_pred))
        val_ccc = calculate_ccc(all_true, all_pred)
        val_acc = binary_accuracy(all_true, all_pred)

        scheduler.step(val_loss)

        # ----- tensorboard -----
        writer.add_scalar("Loss/train_total", train_loss, epoch)
        writer.add_scalar("Loss/train_bce", train_bce, epoch)
        writer.add_scalar("Loss/train_rank", train_rank, epoch)

        writer.add_scalar("Loss/val_total", val_loss, epoch)
        writer.add_scalar("Loss/val_bce", val_bce, epoch)
        writer.add_scalar("Loss/val_rank", val_rank, epoch)

        writer.add_scalar("Metrics/MAE_train", train_mae, epoch)
        writer.add_scalar("Metrics/MAE_val", val_mae, epoch)
        writer.add_scalar("Metrics/CCC_train", train_ccc, epoch)
        writer.add_scalar("Metrics/CCC_val", val_ccc, epoch)
        writer.add_scalar("Metrics/Acc_train", train_acc, epoch)
        writer.add_scalar("Metrics/Acc_val", val_acc, epoch)
        writer.add_scalar("Diagnostics/pred_std", pred_std, epoch)
        writer.add_scalar("Diagnostics/target_std", target_std, epoch)

        writer.add_histogram("Val/predictions", all_pred, epoch)

        for gi, pg in enumerate(optimizer.param_groups):
            writer.add_scalar(f"Train/LR_group{gi}", pg["lr"], epoch)

        print(
            f"época [{epoch+1}/{epochs}]"
            f" | train {train_loss:.4f}"
            f" (bce {train_bce:.4f}, rank {train_rank:.4f})"
            f" | val {val_loss:.4f}"
            f" (bce {val_bce:.4f}, rank {val_rank:.4f})"
            f" | LR {optimizer.param_groups[0]['lr']:.2e}"
        )

        # ----- checkpoint (best + last) + early stopping -----
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "run_name": run_name}, last_path)

        current = {"ccc": val_ccc, "mae": val_mae, "loss": val_loss}[monitor]
        if _is_better(current, best_score, mode):
            best_score = current
            best_metrics = {
                "val_loss": val_loss,
                "val_bce": val_bce,
                "val_rank": val_rank,
                "val_ccc": val_ccc,
                "val_acc": val_acc,
                "epoch": epoch,
            }

            best_true, best_pred = all_true, all_pred
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_metrics": best_metrics,
                "run_name": run_name,
            }, checkpoint_path)
            print(f"  novo melhor ({monitor}={best_score:.4f}) salvo em {checkpoint_path}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"early stopping (sem melhora por {patience} épocas)")
                break

    # scatter do melhor epoch — diagnóstico de variance collapse
    if best_pred is not None:
        _log_pred_scatter(writer, best_true, best_pred, "Val/pred_vs_target", best_metrics["epoch"])

    # hparams para comparar runs na aba HPARAMS (run_name="." liga aos scalars)
    writer.add_hparams(
        {"backbone": backbone_name, "loss": loss_name, "freeze_mode": freeze_mode,
         "freeze_bn_always": freeze_bn_always,
         "lr": LR, "hidden_size": MODEL_DEFAULTS["hidden_size"],
         "num_layers": MODEL_DEFAULTS["num_layers"], "frame_step": MODEL_DEFAULTS["frame_step"]},
        {
            "best/val_loss": best_metrics.get("val_loss", 0.0),
            "best/val_bce": best_metrics.get("val_bce", 0.0),
            "best/val_rank": best_metrics.get("val_rank", 0.0),
            "best/val_ccc": best_metrics.get("val_ccc", 0.0),
            "best/val_acc": best_metrics.get("val_acc", 0.0),
        },
        run_name=".",
    )
    writer.close()
    print(f"concluído. melhor {monitor}: {best_score:.4f}")
    return {"checkpoint": checkpoint_path, "best_metrics": best_metrics}

In [9]:
def run_experiment(
    audiomae,
    alpha=1.0,
    margin_scale=1.0,
    backbone_name="resnet18",
    freeze_mode="finetune",
    unfreeze_epoch=5,
    lr=LR,
    lr_backbone=None,
    weight_decay=WEIGHT_DECAY,
    model_kwargs=None,
    criterion=None,
    epochs=EPOCHS,
    patience=PATIENCE,
    use_amp=False,
):
    """Roda um experimento completo e retorna o resultado."""

    model_kwargs = {**MODEL_DEFAULTS, **(model_kwargs or {})}

    run_name = f"{backbone_name}__{freeze_mode}"

    print(f"\n=== {run_name} ===")

    model = ResNetLSTM_MultiModal(
        audiomae,
        backbone_name=backbone_name,
        **model_kwargs,
    ).to(device)

    if lr_backbone is None:

        optimizer = AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
        )

    else:

        optimizer = AdamW(
            [
                {"params": model.cnn.parameters(), "lr": lr_backbone},
                {"params": model.lstm.parameters(), "lr": lr},
                {"params": model.head.parameters(), "lr": lr},
            ],
            weight_decay=weight_decay,
        )

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=3,
        factor=0.5,
    )

    if criterion is None:
        criterion = CombinedLoss(
            alpha=alpha,
            margin_scale=margin_scale,
        )

    return train_model(
        model=model,
        train_loader=train_multimodal_loader,
        val_loader=valid_multimodal_loader,
        criterion=criterion,
        run_name=run_name,
        optimizer=optimizer,
        scheduler=scheduler,
        backbone_name=backbone_name,
        loss_name=criterion.__class__.__name__,
        freeze_mode=freeze_mode,
        unfreeze_epoch=unfreeze_epoch,
        epochs=epochs,
        patience=patience,
        use_amp=use_amp,
    )

In [10]:
# definindo o modelo para embedding dos mel spectrogramas
model_ae = AutoModel.from_pretrained("hance-ai/audiomae", trust_remote_code=True).to(device)

In [25]:
results = {}

experimentos = (
    [
        ("resnet18", "frozen", 5, 1.0, 1.0),
    ]
    + [
        (backbone, "frozen", 5, 1.0, 1.0)
        for backbone in ["resnet34", "resnet50"]
    ]
    + [
        ("resnet18", "finetune", 5, 1.0, 1.0)
    ]
)

for backbone, freeze_mode, unfreeze_epoch, alpha, margin_scale in experimentos:

    key = (
        f"{backbone}"
        f"__{freeze_mode}"
        f"__a{alpha}"
        f"__m{margin_scale}"
    )

    print(f"\n>>> iniciando {key}")

    try:

        results[key] = run_experiment(
            audiomae=model_ae,
            backbone_name=backbone,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            alpha=alpha,
            margin_scale=margin_scale,
            epochs=100,
        )

    except Exception as e:

        print(f"ERRO em {key}: {e}")
        results[key] = None


print("\n=== RESUMO ===")

for key, result in results.items():

    if result is None:

        print(f"{key}: FALHOU")

    else:

        m = result["best_metrics"]

        print(
            f"{key}"
            f" | loss={m['val_loss']:.4f}"
            f" | BCE={m['val_bce']:.4f}"
            f" | Rank={m['val_rank']:.4f}"
        )


>>> iniciando resnet18__frozen__a1.0__m1.0

=== resnet18__frozen ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs_multimodal_audio/resnet18__frozen_20260617-205600


época 1/100 [treino]:   0%|          | 0/736 [00:00<?, ?it/s]

época 1/100 [val]:   0%|          | 0/295 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x75534ff6a2a0>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ambiente_aluno/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x75534ff6a2a0>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fi

época [1/100] | train 1.2968 (bce 0.6573, rank 0.6395) | val 1.1911 (bce 0.6322, rank 0.5589) | LR 1.00e-04
  novo melhor (loss=1.1911) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet18__frozen.pth


...

época 15/100 [treino]:   0%|          | 0/736 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x75534ff6a2a0>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ambiente_aluno/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x75534ff6a2a0>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fi

época 15/100 [val]:   0%|          | 0/295 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x75534ff6a2a0>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ambiente_aluno/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x75534ff6a2a0>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fi

época [15/100] | train 0.6215 (bce 0.4583, rank 0.1632) | val 1.9305 (bce 0.9686, rank 0.9620) | LR 2.50e-05
early stopping (sem melhora por 10 épocas)
concluído. melhor loss: 0.9678

=== RESUMO ===
resnet18__frozen__a1.0__m1.0 | loss=1.0294 | BCE=0.5821 | Rank=0.4474
resnet34__frozen__a1.0__m1.0 | loss=0.9930 | BCE=0.5633 | Rank=0.4297
resnet50__frozen__a1.0__m1.0: FALHOU
resnet18__finetune__a1.0__m1.0 | loss=0.9678 | BCE=0.5691 | Rank=0.3987


In [11]:
results = {}

experimentos = (
    [
        (backbone, "finetune", 5, 1.0, 1.0) for backbone in ["resnet34", "resnet50"]
    ]
)

for backbone, freeze_mode, unfreeze_epoch, alpha, margin_scale in experimentos:

    key = (
        f"{backbone}"
        f"__{freeze_mode}"
        f"__a{alpha}"
        f"__m{margin_scale}"
    )

    print(f"\n>>> iniciando {key}")

    try:

        results[key] = run_experiment(
            audiomae=model_ae,
            backbone_name=backbone,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            alpha=alpha,
            margin_scale=margin_scale,
            epochs=100,
        )

    except Exception as e:

        print(f"ERRO em {key}: {e}")
        results[key] = None


print("\n=== RESUMO ===")

for key, result in results.items():

    if result is None:

        print(f"{key}: FALHOU")

    else:

        m = result["best_metrics"]

        print(
            f"{key}"
            f" | loss={m['val_loss']:.4f}"
            f" | BCE={m['val_bce']:.4f}"
            f" | Rank={m['val_rank']:.4f}"
        )


>>> iniciando resnet34__finetune__a1.0__m1.0

=== resnet34__finetune ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs_multimodal_audio/resnet34__finetune_20260618-103950


época 1/100 [treino]:   0%|          | 0/736 [00:00<?, ?it/s]

época 1/100 [val]:   0%|          | 0/295 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e0890952200>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ambiente_aluno/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e0890952200>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fi

época [1/100] | train 1.2250 (bce 0.6422, rank 0.5828) | val 1.2112 (bce 0.6779, rank 0.5334) | LR 1.00e-04
  novo melhor (loss=1.2112) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet34__finetune.pth


...

época 22/100 [val]:   0%|          | 0/295 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e0890952200>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ambiente_aluno/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e0890952200>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fi

época [22/100] | train 0.3172 (bce 0.2908, rank 0.0264) | val 1.5607 (bce 0.7191, rank 0.8417) | LR 3.13e-06
early stopping (sem melhora por 20 épocas)
concluído. melhor loss: 1.0559

>>> iniciando resnet50__finetune__a1.0__m1.0

=== resnet50__finetune ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs_multimodal_audio/resnet50__finetune_20260618-133355


época 1/100 [treino]:   0%|          | 0/736 [00:00<?, ?it/s]

ERRO em resnet50__finetune__a1.0__m1.0: CUDA out of memory. Tried to allocate 494.00 MiB. GPU 0 has a total capacity of 19.71 GiB of which 462.19 MiB is free. Including non-PyTorch memory, this process has 18.91 GiB memory in use. Of the allocated memory 18.48 GiB is allocated by PyTorch, and 192.72 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

=== RESUMO ===
resnet34__finetune__a1.0__m1.0 | loss=1.0559 | BCE=0.5992 | Rank=0.4567
resnet50__finetune__a1.0__m1.0: FALHOU


In [11]:
results = {}

experimentos = (
    [
        ("resnet50", "finetune", 5, 1.0, 1.0)
    ]
)

for backbone, freeze_mode, unfreeze_epoch, alpha, margin_scale in experimentos:

    key = (
        f"{backbone}"
        f"__{freeze_mode}"
        f"__a{alpha}"
        f"__m{margin_scale}"
    )

    print(f"\n>>> iniciando {key}")

    try:

        results[key] = run_experiment(
            audiomae=model_ae,
            backbone_name=backbone,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            alpha=alpha,
            margin_scale=margin_scale,
            epochs=100,
        )

    except Exception as e:

        print(f"ERRO em {key}: {e}")
        results[key] = None


print("\n=== RESUMO ===")

for key, result in results.items():

    if result is None:

        print(f"{key}: FALHOU")

    else:

        m = result["best_metrics"]

        print(
            f"{key}"
            f" | loss={m['val_loss']:.4f}"
            f" | BCE={m['val_bce']:.4f}"
            f" | Rank={m['val_rank']:.4f}"
        )


>>> iniciando resnet50__finetune__a1.0__m1.0

=== resnet50__finetune ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs_multimodal_audio/resnet50__finetune_20260618-140929


época 1/100 [treino]:   0%|          | 0/1472 [00:00<?, ?it/s]

época 1/100 [val]:   0%|          | 0/589 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x784a51d562a0>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ambiente_aluno/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x784a51d562a0>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fi

época [1/100] | train 1.4057 (bce 0.7100, rank 0.6957) | val 1.5148 (bce 0.7664, rank 0.7485) | LR 1.00e-04
  novo melhor (loss=1.5148) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet50__finetune.pth


...

época 22/100 [treino]:   0%|          | 0/1472 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x784a51d562a0>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ambiente_aluno/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x784a51d562a0>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fi

época 22/100 [val]:   0%|          | 0/589 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x784a51d562a0>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ambiente_aluno/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x784a51d562a0>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fi

época [22/100] | train 0.1827 (bce 0.1827, rank 0.0000) | val 2.7671 (bce 1.2544, rank 1.5126) | LR 3.13e-06
early stopping (sem melhora por 20 épocas)
concluído. melhor loss: 1.3627

=== RESUMO ===
resnet50__finetune__a1.0__m1.0 | loss=1.3627 | BCE=0.6834 | Rank=0.6793


In [26]:
%load_ext tensorboard
%tensorboard --logdir /mnt/storage_C4/gaussian_football/runs_multimodal_audio/